# DEMO: Monitoring Dashboard Usage with Audit Logs

In Databricks, you have something powerful for monitoring: **system tables** and **audit logs** — queryable with SQL.

| Databricks |
| --- |
| `system.access.audit` table (ALL workspace events) |
| Every action logged (create, view, edit, publish, embed) |
| Write your own monitoring queries |
| Configurable retention |
| Join with any other table (it's just SQL!) |

**Key advantage:** In Databricks, monitoring IS analytics. You query system tables with the same SQL you use for everything else. You can even build dashboards on top of your monitoring data!

In this demo, we'll:
1. Understand the `system.access.audit` table structure
2. Find the most-viewed dashboards
3. Track dashboard creation activity
4. See who viewed a specific dashboard
5. Monitor embedded dashboard access

> **Note:** Run the setup cell below. The `system.access.audit` table requires account admin or metastore admin role. If you don't have access, follow along with the instructor.

---

In [0]:
%run ./Setup

## The `system.access.audit` Table

This is the master audit log. **Every action** in your workspace is recorded here.

| Column | What It Contains |
| --- | --- |
| `event_time` | When the action occurred (timestamp) |
| `event_date` | Date partition (use for efficient filtering) |
| `user_identity.email` | Who performed the action |
| `action_name` | What they did (e.g., `getDashboard`, `createDashboard`) |
| `request_params` | Details as a map (dashboard_id, settings, etc.) |
| `source_ip_address` | Where they connected from |
| `service_name` | Which Databricks service was used |

### Dashboard-related action names:

| Action | Meaning |
| --- | --- |
| `createDashboard` | A new dashboard was created |
| `getDashboard` | A draft dashboard was viewed |
| `getPublishedDashboard` | A published dashboard was viewed |
| `updateDashboard` | A dashboard was edited |
| `publishDashboard` | A dashboard was published |
| `deleteDashboard` | A dashboard was deleted |

> **Requires:** Account admin + metastore admin role, or explicit GRANT to the `system.access` schema.

---

## Query 1: Find the Most-Viewed Dashboards

This tells you which dashboards are getting the most traffic — useful for prioritizing optimization and maintenance.

---

In [0]:
%sql
-- ============================================================
-- Most-Viewed Dashboards (Last 30 Days)
-- ============================================================
-- NOTE: Requires access to system.access.audit

SELECT
  request_params.dashboard_id AS dashboard_id,
  COUNT(*) AS view_count,
  COUNT(DISTINCT user_identity.email) AS unique_viewers
FROM system.access.audit
WHERE action_name IN ('getDashboard', 'getPublishedDashboard')
  AND event_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY request_params.dashboard_id
ORDER BY view_count DESC
LIMIT 10;

## Query 2: Dashboard Creation Activity

Track how many dashboards are being created — useful for measuring platform adoption.

---

In [0]:
%sql
-- ============================================================
-- Dashboard Creation Activity (Last 7 Days)
-- ============================================================
-- Track platform adoption: who's building dashboards?

SELECT
  event_date,
  user_identity.email AS creator,
  COUNT(*) AS dashboards_created
FROM system.access.audit
WHERE action_name = 'createDashboard'
  AND event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY event_date, user_identity.email
ORDER BY event_date DESC;

## Query 3: Who Viewed a Specific Dashboard?

Answer the question: "Is anyone actually using this dashboard I built?"

Replace the dashboard_id below with your actual dashboard ID from the setup output.

---

In [0]:
%sql
-- ============================================================
-- Who Viewed a Specific Dashboard? (Last 30 Days)
-- ============================================================
-- Replace <YOUR_DASHBOARD_ID> with the ID from setup output.

SELECT
  event_date,
  user_identity.email AS viewer,
  action_name,
  source_ip_address
FROM system.access.audit
WHERE action_name IN ('getDashboard', 'getPublishedDashboard')
  AND request_params.dashboard_id = '<YOUR_DASHBOARD_ID>'
  AND event_date >= CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC
LIMIT 50;

## Query 4: Monitor Embedded Dashboard Access

Track which dashboards have been embedded and from where. Important for security and compliance teams.

---

In [0]:
%sql
-- ============================================================
-- Monitor Embedded Dashboard Access
-- ============================================================
-- Track dashboards accessed via embedding (iframes).
-- Useful for security/compliance auditing.

SELECT
  request_params.settingTypeName AS setting_type,
  source_ip_address,
  user_identity.email AS user_email,
  action_name,
  event_date
FROM system.access.audit
WHERE request_params.settingTypeName ILIKE 'aibi%'
  AND event_date >= CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_date DESC
LIMIT 20;

## Bonus: Build a Monitoring Dashboard

Since audit data is just another SQL table, you can build a **dashboard on top of your monitoring data**:

1. Create a new dashboard
2. Add these queries as datasets
3. Build visualizations: line chart of daily views, table of top dashboards, counter of unique viewers
4. Schedule it daily
5. Subscribe your admin team

This is monitoring-as-analytics — the same tools you use for business data work for operational data too.

### Example monitoring dashboard datasets:

| Dataset | Query Logic | Widget Type |
| --- | --- | --- |
| Daily Views | COUNT by event_date | Line chart |
| Top 10 Dashboards | GROUP BY dashboard_id, count views | Bar chart |
| Active Users | COUNT DISTINCT user_identity.email | Counter |
| Creation Trend | COUNT createDashboard by week | Area chart |


---

---

## Summary

| What You Learned | Key Takeaway |
| --- | --- |
| `system.access.audit` | Master audit log — every workspace action recorded |
| Dashboard views | Filter by `getDashboard` / `getPublishedDashboard` |
| Creation tracking | Filter by `createDashboard` → measure adoption |
| Per-dashboard analysis | Filter by `request_params.dashboard_id` |
| Embedded monitoring | Filter by `settingTypeName ILIKE 'aibi%'` |
| Monitoring dashboards | Build dashboards on audit data (meta-dashboards!) |

### Access Requirements

| Role | Access Level |
| --- | --- |
| Account admin | Full access to all system tables |
| Metastore admin | Full access to all system tables |
| Other users | Need explicit `GRANT SELECT ON system.access.audit` |